In [22]:
from enum import unique
from unittest.util import strclass
from pytz import utc
from ticktick.oauth2 import OAuth2        # OAuth2 Manager
from ticktick.api import TickTickClient   # Main Interface
from os import environ
from dotenv import load_dotenv
import json
import os
from datetime import datetime, timedelta,timezone
import logging

# setup 
load_dotenv('../.secrets')
client_id=environ.get('client_id')
client_secret=environ.get('client_secret')
username=environ.get('username')
password=environ.get('password')
redirect_uri=environ.get('redirect_uri')

auth_client = OAuth2(client_id=client_id,
                     client_secret=client_secret,
                     redirect_uri=redirect_uri)

client = TickTickClient(username, password, auth_client)



# default_start = datetime(2022, 7, 23,tzinfo=timezone.utc)
default_start = datetime(2024, 2, 10,tzinfo=timezone.utc)
tasks_file_path = 'raw/all_tasks.json'
completed_tasks_file_path= 'raw/completed_tasks.json'
new_tasks_file_path = 'raw/new_tasks.json'
lists_file_path = 'raw/all_lists.json'
folders_file_path = 'raw/all_folders.json'
date_format = '%Y-%m-%dT%H:%M:%S.%f%z'


logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s'
)

def deduplicate(source) -> list:
    """
    checks each item and remove duplicated
    """
    # unique_items={}
    # unique_list=[]

    # for item in source:
    #     item_id=item.get("id")

    #     if item_id not in unique_items:
    #         unique_list.append(item)
    #         unique_items[item_id]=True
    # return unique_list
    pass


def _get_completed_tasks(start=None, end=datetime.now(timezone.utc), full_load=True):
    """_summary_
    returns a json string
    internal func - uses tickpy to grab completed tasks from start > end.
    `start` has a default value 2022/07/23 which is the start of my ticktick interactions
    `end` defaults to the runtime date.

    Returns:
        list: all the completed tasks in the interval.
    """
    
    completed_tasks=[] 
    logging.info('start loading tasks')
    if full_load:
        current_date=default_start
    elif not full_load: 
        current_date=start
    while current_date <= end:
        tasks=client.task.get_completed(current_date)
        if tasks != []:
            for task in tasks:
                completed_tasks.append(task)
            logging.info(f'loaded {len(tasks)} new tasks from {current_date}. next interation...')
        current_date += timedelta(days=1)
    # completed_tasks=json.dumps(completed_tasks)
    return completed_tasks

def get_completed_task() -> list:
    """
    returns a list the full completed tasks and utilize existing file as cache if available.
    """
    try:
        with open(completed_tasks_file_path,'r') as f:
            cached=json.load(f)
            cached_completed=[item for item in cached if 'completedTime' in item]
            last_cached_date=cached_completed[-1]['completedTime']
            last_cached_date=datetime.strptime(last_cached_date,date_format)
            last_cached_date=last_cached_date - timedelta(days=1)
            full_load=False
    except FileNotFoundError:
        logging.info('no cache found. doing full load...')
        cached=[]
        last_cached_date=None
        full_load=True
    
    # checks existing and append to cached list 
    net_new=_get_completed_tasks(start=last_cached_date,full_load=full_load)
    # net_new=deduplicate(net_new)

    # cached=deduplicate(cached)

    # concatenate final completed list
    all_completed_tasks=net_new+cached
    return all_completed_tasks
    

def get_new_tasks() -> list:
    new_tasks=client.state['tasks']
    return new_tasks

def get_all_tasks() -> list:
    new=get_new_tasks()
    completed=get_completed_task()
    all_tasks=new+completed
    return all_tasks

def get_lists_and_folders():
    lists = client.state['projects']
    folders = client.state['project_folders']
    return lists, folders

def dump_to_file(source:list, target:str):
    """
    takes source then dumps to json raw file 
    """

    with open(target,'w') as f:
        json.dump(source,f,indent=4,)

lists,folders = get_lists_and_folders()
tasks=get_all_tasks()





2024-02-12 22:02:19,078 - INFO - start loading tasks
2024-02-12 22:02:19,315 - INFO - loaded 80 new tasks from 2024-02-10 17:35:30+00:00. next interation...
2024-02-12 22:02:19,548 - INFO - loaded 5 new tasks from 2024-02-11 17:35:30+00:00. next interation...


In [16]:
dump_to_file(get_completed_task(),completed_task_file_path)

2024-02-12 21:52:25,574 - INFO - no cache found. doing full load...
2024-02-12 21:52:25,575 - INFO - start loading tasks
2024-02-12 21:52:25,794 - INFO - loaded 80 new tasks from 2024-02-10 00:00:00+00:00. next interation...
2024-02-12 21:52:26,048 - INFO - loaded 5 new tasks from 2024-02-11 00:00:00+00:00. next interation...
2024-02-12 21:52:26,252 - INFO - loaded 7 new tasks from 2024-02-12 00:00:00+00:00. next interation...


In [14]:
dump_to_file(get_new_tasks(),new_tasks_file_path)

In [29]:
done=get_new_tasks()

In [31]:
with open(tasks_file_path,'r') as f:
    task=json.load(f)

In [32]:
len(task)

1621